# Demostración

**Fase:** E20

## Objetivo

Mostrar el mismo caso con y sin la evidencia relevante en el corpus, y contrastar explicación trazable frente a abstención.

## Relación con la pregunta de investigación

> ¿Puede un agente con RAG explicar el diagnóstico de un vehículo a partir de datos OBD-II, rastreando cada explicación hasta su evidencia de origen, y reconocer cuándo la evidencia disponible no es suficiente para sostener una explicación en lugar de generar una hipótesis sin fundamento?

Es la escenificación directa de la PI ante el tribunal.

## Hipótesis

_Por declarar antes de ejecutar la fase._

## Datos

_Dataset, versión y partición utilizados._

## Método

_Procedimiento, parámetros y semillas._

## Métricas

_Qué se mide y con qué instrumento._

## Criterio de éxito o refutación

_Fijado antes de mirar los resultados._


In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd

from mecanico_ia_exp.config import paths


## Demostración

La cadena completa sobre un caso, y el mismo caso sin su evidencia.

In [2]:
import json
s = json.loads((paths.EXPERIMENTS / 'E15_suficiencia.json').read_text())
caso = next(c for c in s['cases'] if c['case_id'] == 'C05_vacuum_leak')
b = caso['bundle']
print(f"vehículo   : {b['vehicle_id']}")
print(f"diagnóstico: {b['predicted_class']}")
print(f"confianza  : {b['calibrated_confidence']}")
print()
print('EVIDENCIA OBSERVACIONAL')
for o in b['observations'][:3]:
    print(f"  {o['observation_id']}: {o['signal']}.{o['statistic']} = "
          f"{o['value']:.4g} ({o['direction']}) · ventana "
          f"{o['window_start_s']:.0f}-{o['window_end_s']:.0f} s")

vehículo   : mazda-3-bk-2007
diagnóstico: F01_vacuum_leak
confianza  : 0.9078

EVIDENCIA OBSERVACIONAL
  OBS001: ltft_bank1.max = 2.716 (increases) · ventana 120-239 s
  OBS002: stft_bank1.mean = 7.077 (increases) · ventana 120-239 s
  OBS003: ltft_bank1.mean = 1.294 (increases) · ventana 120-239 s


In [3]:
print('EVIDENCIA DOCUMENTAL')
for d in b['documents'][:3]:
    print(f"  {d['chunk_id'][-10:]} · {d['section']}")
    print(f"      {d['publisher']} · pág {d['page_start']} · "
          f"DTC {d['dtc_codes']} · aplicabilidad {d['applicability']}")

EVIDENCIA DOCUMENTAL
  951:001399 · DTC DETECTION LOGIC AND CONDITIONS [ZJ, Z6]
      Mazda Motor Corporation · pág 7 · DTC ['P2178', 'P2187', 'P2188', 'P2195'] · aplicabilidad COMPATIBLE
  951:001359 · DTC DETECTION LOGIC AND CONDITIONS [LF]
      Mazda Motor Corporation · pág 7 · DTC ['P2187', 'P2188', 'P2195'] · aplicabilidad COMPATIBLE
  951:001851 · DTC P2187 [ZJ, Z6]
      Mazda Motor Corporation · pág 1 · DTC ['P2187'] · aplicabilidad EXACT_MATCH


In [4]:
print('DECISIÓN:', caso['verdict']['decision'])
print()
ausente = next(c for c in s['cases'] if 'retirada' in c['case_id'])
print('Mismo caso, retirando la evidencia del índice:')
print('  decisión:', ausente['verdict']['decision'])
print('  motivos :', ausente['verdict']['reasons'])

DECISIÓN: EXPLAIN

Mismo caso, retirando la evidencia del índice:
  decisión: ABSTAIN
  motivos : ['NO_APPLICABLE_DOCUMENTARY_EVIDENCE', 'LOW_CLAIM_COVERAGE']


**Ésa es la respuesta a la pregunta de investigación**, sobre datos sintéticos calibrados: cuando la evidencia está, el agente explica trazando cada afirmación; cuando no está, se abstiene.

## Resultados

_Lo observado, sin interpretación._

## Interpretación

_Qué significa, y qué no significa._

## Limitaciones

_Qué queda fuera del alcance de lo mostrado._

## Conclusión experimental

_Respuesta al criterio fijado en la cabecera._

## Artefactos generados

_Ficheros producidos, con su ruta._
